# 1. Khởi tạo môi trường & Đọc dữ liệu topic

In [ ]:
# export
import json
import random
import time
from datetime import datetime

import pandas as pd

from helpers.crawl_trend_data import crawl_trend_data
from helpers.get_configs import get_config

TOPICS_PATH = get_config('dirs.yaml', ['raw_data', 'trend_topics'], as_path=True)
TRENDS_PATH = get_config('dirs.yaml', ['raw_data', 'trends'], as_path=True)
SCRIPT_OUTPUT_PATH = get_config('dirs.yaml', ['unit_scripts', 'get_trends'], as_path=True)

# Chỉ đọc 3 cột cần thiết (file trend_topics có rất nhiều cột)
df_topics = pd.read_csv(TOPICS_PATH, encoding='utf-8', usecols=['keyword', 'keyword_type', 'mid'])

In [ ]:
# Khảo sát thông tin tổng quan của dữ liệu
df_topics.info()
df_topics.head()

# 2. Chuẩn bị danh sách cần cào

- Bỏ dòng thiếu `mid`, loại trùng theo cặp (`keyword`, `keyword_type`, `mid`).
- Một `mid` có thể xuất hiện ở nhiều keyword; mỗi `mid` chỉ cào **một lần** (cache), sau đó gắn lại `keyword` / `keyword_type` tương ứng cho từng dòng.

In [ ]:
# export
df_topics = (
    df_topics.dropna(subset=['mid'])
    .assign(mid=lambda d: d['mid'].astype(str).str.strip())
    .query("mid != ''")
    .drop_duplicates(subset=['keyword', 'keyword_type', 'mid'])
    .reset_index(drop=True)
)

tasks = list(df_topics[['keyword', 'keyword_type', 'mid']].itertuples(index=False, name=None))
unique_mids = df_topics['mid'].nunique()

print(f"Số dòng cần xử lý: {len(tasks)} | Số mid unique (số lần cào thực tế): {unique_mids}")

# 3. Cấu hình & hàm cào

- `date = '2010-01-01%202026-12-31'`, `geo = 'VN'`, `q = mid`.
- Kết quả của `crawl_trend_data` (`GoogleTrendResult`) được chuyển sang dict bằng `.to_dict()`, rồi thêm 2 thuộc tính `keyword` và `keyword_type`.
- Cào xong toàn bộ mới lưu một lần ra file JSON.

In [ ]:
# export
DATE = '2010-01-01%202026-12-31'
GEO = 'VN'

SLEEP_RANGE = (2.0, 5.0)        # nghỉ ngẫu nhiên giữa 2 lần cào (giây)
CONSECUTIVE_FAIL_LIMIT = 3
COOLDOWN_SECONDS = 120          # nghỉ dài khi thất bại liên tiếp (nghi bị chặn / rate limit)


def save_trends(records, path):
    """Ghi array of object ra file JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def fetch_trend(mid):
    """Cào 1 mid, luôn trả về dict (kể cả khi lỗi) để pipeline không bị dừng giữa chừng."""
    try:
        return crawl_trend_data(q=mid, date=DATE, geo=GEO).to_dict()
    except Exception as e:
        return {
            'query_terms': [mid],
            'date_range': DATE,
            'geo': GEO,
            'scraped_at': datetime.utcnow().isoformat(),
            'status': 'failed',
            'errors': [repr(e)],
            'metadata': {},
            'interest_over_time': None,
            'interest_by_region': None,
            'commonly_searched_queries': None,
        }

# 4. Cào dữ liệu Google Trends

In [ ]:
# export
records = []
mid_cache = {}                  # mid -> dict kết quả (tránh cào lại cùng 1 mid)
consecutive_fails = 0

for i, (keyword, keyword_type, mid) in enumerate(tasks, start=1):
    from_cache = mid in mid_cache
    if from_cache:
        data = json.loads(json.dumps(mid_cache[mid]))   # bản sao độc lập
    else:
        data = fetch_trend(mid)

    data['keyword'] = keyword
    data['keyword_type'] = keyword_type
    records.append(data)

    if from_cache:
        print(f"[{i}/{len(tasks)}] ({keyword_type}) {keyword} | {mid} -> dùng cache")
        continue

    status = data.get('status')
    print(f"[{i}/{len(tasks)}] ({keyword_type}) {keyword} | {mid} -> {status}")

    if status == 'failed':
        consecutive_fails += 1
        if consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
            print(f"  Thất bại liên tiếp {consecutive_fails} lần -> nghỉ {COOLDOWN_SECONDS}s")
            time.sleep(COOLDOWN_SECONDS)
            consecutive_fails = 0
    else:
        mid_cache[mid] = data
        consecutive_fails = 0

    time.sleep(random.uniform(*SLEEP_RANGE))

save_trends(records, TRENDS_PATH)
print(f"Hoàn tất. Tổng số object: {len(records)} -> {TRENDS_PATH}")

# 5. Kiểm tra kết quả

In [ ]:
# Thống kê trạng thái cào
df_status = pd.DataFrame({
    'keyword_type': [r['keyword_type'] for r in records],
    'status': [r['status'] for r in records],
})
print(df_status['status'].value_counts())
print(df_status.groupby(['keyword_type', 'status']).size().unstack(fill_value=0))

In [ ]:
# Xem nhanh 1 object & các mid thất bại
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:2000])

failed = [(r['keyword'], r['keyword_type'], r['query_terms'][0], r['errors']) for r in records if r['status'] == 'failed']
print(f"Số object thất bại: {len(failed)}")
failed[:10]

# 6. Xuất Notebook sang Unit Script

In [ ]:
from helpers.export_to_script import export_to_script

export_to_script(output_path=SCRIPT_OUTPUT_PATH)